In [1]:
from datasets import load_dataset
import pandas as pd
from collections import defaultdict, Counter
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import json
import re
from torch.utils.data import TensorDataset, DataLoader
import kenlm
from tqdm import tqdm
import math
import lightgbm as lgb
import warnings
import optuna

from itertools import chain
from concurrent.futures import ThreadPoolExecutor, as_completed
import random
from sklearn.metrics.pairwise import cosine_similarity
import time
import psutil
import os
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import torch.nn.functional as F
from collections import Counter

C:\Users\ADMIN\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [3]:
vocab = []
with open(r"D:\NLP\project\vocabulary.txt", 'r', encoding='utf-8') as f:
    vocab = f.read().splitlines()

# Lọc các từ giống nhau
vocab = list(dict.fromkeys(vocab))

# Ánh xạ word và idx
word_to_idx = {word: i for i, word in enumerate(vocab)}

In [4]:
dataset = load_dataset("yammdd/vietnamese-error-correction-corpus")

In [5]:
def process_dataset(examples):
    inputs = []
    targets = []

    for inp, tgt in zip(examples['input'], examples['target']):
        # Viết thường
        inp_str = str(inp).lower()
        tgt_str = str(tgt).lower()

        # Xóa dấu câu và xóa chữ số
        inp_clean = re.sub(r'[^\w\s_]|\d+', '', inp_str)
        tgt_clean = re.sub(r'[^\w\s_]|\d+', '', tgt_str)

        inp_tokens = inp_clean.split()
        tgt_tokens = tgt_clean.split()

        # Kiểm tra điều kiện độ dài: Chỉ giữ lại nếu bằng nhau
        if len(inp_tokens) != len(tgt_tokens) or len(inp_tokens) == 0:
          continue

        # Thay thế các từ không phải tiếng Việt mà bị lỗi
        new_inp_tokens = []
        for inp_word, tgt_word in zip(inp_tokens, tgt_tokens):
            if tgt_word not in word_to_idx:
                new_inp_tokens.append(tgt_word)
            else:
                new_inp_tokens.append(inp_word)

        inputs.append(" ".join(new_inp_tokens))
        targets.append(" ".join(tgt_tokens))

    return {"input": inputs, "target": targets}

In [6]:
df = dataset.map(process_dataset,
                batched=True, 
                remove_columns=dataset['train'].column_names)

In [7]:
VIETNAMESE_DIACRITICS = re.compile(r'[àáảãạâầấẩẫậăằắẳẵặèéẻẽẹêềếểễệđìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵ]')

def split_data(dataset):
    # Lỗi chính tả
    def error_1(example):
        inp = str(example['input'])

        return bool(VIETNAMESE_DIACRITICS.search(inp))

    # Lỗi không dấu
    def error_2(example):
        inp = str(example['input'])

        return not bool(VIETNAMESE_DIACRITICS.search(inp))

    # Lọc song song trên cả 3 tập (train, validation, test) của DatasetDict lỗi
    df_error1 = dataset.filter(error_1)
    df_error2 = dataset.filter(error_2)
    
    return df_error1, df_error2

In [8]:
df1, df2 = split_data(df)

df1_train = pd.DataFrame(df1['train'])
df1_test = pd.DataFrame(df1['test'])
df1_valid = pd.DataFrame(df1['validation'])

df2_train = pd.DataFrame(df2['train'])
df2_test = pd.DataFrame(df2['test'])
df2_valid = pd.DataFrame(df2['validation'])

In [12]:
abbreviation_dict = {}

with open(r"D:\NLP\project\teen_code.txt", 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        
        parts = line.split(maxsplit=1)
        shortcut, full_word = parts[0].lower(), parts[1].lower()
        abbreviation_dict[shortcut] = full_word

In [13]:
def replace_abbreviations(sentence):
    words = sentence.lower().split()
    
    for i, word in enumerate(words):
        if word in abbreviation_dict:
            if len(abbreviation_dict[word].split()) == 1:
                words[i] = abbreviation_dict[word]
            
    return " ".join(words)

In [15]:
def find_misspelled_words_and_targets(input_sentence, target_sentence):
    input_tokens = input_sentence.split()
    target_tokens = target_sentence.split()
    
    error_indices = []
    pairs = []
    if len(input_tokens) != len(target_tokens):
        return [], []

    for i in range(len(input_tokens)):
        if input_tokens[i] != target_tokens[i] and target_tokens[i] in word_to_idx:
            pairs.append((input_tokens[i], target_tokens[i]))
            error_indices.append(i)
            
    return pairs, error_indices


In [10]:
def detect_error_tune(sentence, model_lm, alpha, hard_ceiling, hard_floor):
    scores = list(model_lm.full_scores(sentence))[:-1]
    words = sentence.split()

    error_indices = set()
    valid_probs = []
    valid_indices = []

    for i, (prob, length, is_oov) in enumerate(scores):
        if i >= len(words):
            continue

        # Lọc các từ out of vocabulary
        if is_oov:
            error_indices.add(i)
            
        valid_probs.append(prob)
        valid_indices.append(i)

    # Ngưỡng ken_lm nhận từ tham số tối ưu hóa bên ngoài
    if valid_probs:
        mean_prob = np.mean(valid_probs)
        std_prob = np.std(valid_probs) 
        
        dynamic_threshold = mean_prob - (alpha * std_prob)

        for idx, prob in zip(valid_indices, valid_probs):
            is_anomaly = (prob < dynamic_threshold) and (prob < hard_ceiling)
            is_absolute_error = (prob < hard_floor)
            
            if is_anomaly or is_absolute_error:
                error_indices.add(idx)

    return sorted(list(error_indices))


In [28]:
valid_cached_dataset = []

for idx, row in df1_valid.iterrows():
    input_sent = str(row['input'])
    target_sent = str(row['target'])

    # Thay thế các từ viết tắt
    input_sent = replace_abbreviations(input_sent)

    # Tìm các từ lỗi thực tế theo hàm gốc của bạn
    error_pairs, error_indices = find_misspelled_words_and_targets(input_sent, target_sent)
    
    # Giữ nguyên logic: Bỏ qua câu không có lỗi từ code gốc của bạn
    if not error_pairs:
        continue 
        
    valid_cached_dataset.append((input_sent, error_indices))

print(len(valid_cached_dataset))

2663


In [ ]:
def objective_f05(trial):
    # 1. Định nghĩa khoảng không gian quét cho 3 tham số
    alpha = trial.suggest_float('alpha', 1.0, 10.0, step=0.1)
    hard_ceiling = trial.suggest_float('hard_ceiling', -10, -2, step=0.01)
    hard_floor = trial.suggest_float('hard_floor', -10, -2, step=0.01)
    
    # Ràng buộc logic toán học: trần luôn lớn hơn sàn về giá trị âm
    if hard_ceiling <= hard_floor:
        return 0.0

    # 2. Khởi tạo lại bộ đếm tổng từ code số 3 của bạn
    total_TP = 0  
    total_FP = 0  
    total_FN = 0  

    # 3. Vòng lặp tính toán TP, FP, FN trên tập data đã lọc
    for input_sent, error_indices in valid_cached_dataset:
        
        # Gọi hàm detect_error đã tinh chỉnh tham số động
        error_detect = detect_error_tune(input_sent, alpha, hard_ceiling, hard_floor)

        set_true = set(error_indices)
        set_pred = set(error_detect)

        # Tính TP, FP, FN (Giữ nguyên logic gốc của bạn)
        TP = len(set_true & set_pred)
        FP = len(set_pred - set_true)
        FN = len(set_true - set_pred)

        # Cộng dồn vào tổng
        total_TP += TP
        total_FP += FP
        total_FN += FN

    # 4. Tính toán Precision, Recall và F0.5 để trả về cho Optuna tối ưu
    precision = total_TP / (total_TP + total_FP) if (total_TP + total_FP) > 0 else 0
    recall = total_TP / (total_TP + total_FN) if (total_TP + total_FN) > 0 else 0
    f05_score = (1 + 0.5**2) * (precision * recall) / ((0.5**2 * precision) + recall)
    
    return f05_score

In [24]:
model_lm = kenlm.Model(r"D:/NLP/project/trigram.bin")

In [ ]:
study = optuna.create_study(direction="maximize")

# n_trials=40 lượt rà soát liên tục là vừa đủ
study.optimize(objective_f05, n_trials=40, show_progress_bar=True)

print(f"Điểm F0.5 cao nhất tìm được: {study.best_value * 100:.2f}%")
print(f"Bộ tham số tối ưu tuyệt đối: {study.best_params}")

Best trial: 67. Best value: 0.794013: 100%|██████████| 100/100 [00:08<00:00, 11.98it/s]

Điểm F0.5 cao nhất tìm được: 79.40%
Bộ tham số tối ưu tuyệt đối: {'alpha': 4.6, 'hard_ceiling': -3.8999999999999995, 'hard_floor': -6.26}
